Looks like it possibly connects but isn't listening properly, will need to see if it can physically move the syringe to see if it is actually connecting

In [1]:
import ftd2xx
print(ftd2xx.listDevices())

[b'G0SX4P41']


In [2]:
import ftd2xx
import time

class Gilson4020PumpUSB:
    def __init__(self, ftdi_serial=None):
        self.device = self._connect_to_device(ftdi_serial)
        self._configure_device()

    def _connect_to_device(self, ftdi_serial):
        devices = ftd2xx.listDevices()
        if not devices:
            raise RuntimeError("No FTDI devices found.")

        decoded_devices = [d.decode() if isinstance(d, bytes) else d for d in devices]
        if ftdi_serial is None:
            ftdi_serial = decoded_devices[0]
        elif ftdi_serial not in decoded_devices:
            raise RuntimeError(f"FTDI device with serial '{ftdi_serial}' not found. Available: {decoded_devices}")

        try:
            return ftd2xx.openEx(ftdi_serial.encode('ascii'), 1)
        except ftd2xx.DeviceError as e:
            raise RuntimeError(f"Failed to open FTDI device '{ftdi_serial}': {e}")

    def _configure_device(self):
        self.device.setTimeouts(1000, 1000)
        self.device.purge()
        self.device.setDtr()
        self.device.setRts()
        self.device.setLatencyTimer(16)
        self.device.setUSBParameters(4096, 4096)
        self.device.setChars(0, False, 0, False)
        self.device.setFlowControl(0x0100, 0x11, 0x13)
        self.device.setBitMode(0, 0x00)
        self.device.setBitMode(0, 0x40)
        # self.device.setEcho(False)

    def send_command(self, cmd, read_delay=0.2, max_attempts=10):
        self.device.write((cmd + "\r").encode('ascii'))
        time.sleep(read_delay)
    
        response = b''
        for _ in range(max_attempts):
            chunk = self.device.read(1024)
            if chunk:
                response += chunk
                if b'\r' in chunk or b'\n' in chunk:
                    break
            else:
                time.sleep(0.1)
    
        return response.decode('ascii', errors='ignore').strip()


    def identify(self):
        return self.send_command('%')

    def home(self):
        return self.send_command('p')

    def get_serial_number(self):
        self.send_command('.sn')
        time.sleep(0.2)
        return self.send_command('.')

    def reset(self):
        return self.send_command('$')

    def close(self):
        self.device.close()


In [ ]:
pump = Gilson4020PumpUSB()

In [3]:
print("Identify:", pump.identify())
print("Serial:", pump.get_serial_number())
print("Home:", pump.home())
#pump.close()

Identify: 
Serial: 
Home: 


In [4]:
pump.close()